<a href="https://colab.research.google.com/github/khushi-2003/AI-projects/blob/main/TextGenerationModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Text Generation Model
1. IMPORT LIBRARIES

In [5]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping

Data Gathering

In [6]:
with open("/content/alllines.txt", "r", encoding="utf-8") as file:
    text = file.read()

print(text[:500])

"ACT I"
"SCENE I. London. The palace."
"Enter KING HENRY, LORD JOHN OF LANCASTER, the EARL of WESTMORELAND, SIR WALTER BLUNT, and others"
"So shaken as we are, so wan with care,"
"Find we a time for frighted peace to pant,"
"And breathe short-winded accents of new broils"
"To be commenced in strands afar remote."
"No more the thirsty entrance of this soil"
"Shall daub her lips with her own children's blood,"
"Nor more shall trenching war channel her fields,"
"Nor bruise her flowerets with the ar


In [7]:
print(len(text))

4583798


Tokenization

In [8]:
text = text[:50000]
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
print("Vocabulary Size:", total_words)

# Reverse mapping: index -> word
index_word = {i: word for word, i in tokenizer.word_index.items()}


Vocabulary Size: 2094


In [9]:
tokenizer.word_index

{'the': 1,
 'and': 2,
 'of': 3,
 'i': 4,
 'a': 5,
 'to': 6,
 'in': 7,
 'you': 8,
 'my': 9,
 'not': 10,
 'for': 11,
 'thou': 12,
 'be': 13,
 'that': 14,
 'with': 15,
 'he': 16,
 'as': 17,
 'it': 18,
 'this': 19,
 'me': 20,
 'is': 21,
 'will': 22,
 'but': 23,
 'what': 24,
 'have': 25,
 'his': 26,
 'shall': 27,
 'by': 28,
 'so': 29,
 'him': 30,
 'we': 31,
 'all': 32,
 'your': 33,
 'lord': 34,
 'thee': 35,
 "i'll": 36,
 'no': 37,
 'them': 38,
 'an': 39,
 'when': 40,
 'they': 41,
 'are': 42,
 'our': 43,
 'good': 44,
 'if': 45,
 'on': 46,
 'then': 47,
 'or': 48,
 'at': 49,
 'thy': 50,
 'now': 51,
 'well': 52,
 'us': 53,
 'upon': 54,
 'francis': 55,
 'enter': 56,
 'there': 57,
 'come': 58,
 'do': 59,
 'am': 60,
 'sir': 61,
 'did': 62,
 "'": 63,
 'king': 64,
 'from': 65,
 'more': 66,
 'which': 67,
 'hath': 68,
 'let': 69,
 'than': 70,
 'why': 71,
 'poins': 72,
 'go': 73,
 'o': 74,
 'give': 75,
 'time': 76,
 'was': 77,
 'ye': 78,
 'mortimer': 79,
 'such': 80,
 'where': 81,
 'horse': 82,
 'see':

CREATE INPUT SEQUENCES : Instead of splitting line by line, use continuous text for better learning

In [10]:
text[:50]

'"ACT I"\n"SCENE I. London. The palace."\n"Enter KING'

In [11]:
token_list = tokenizer.texts_to_sequences([text])[0]

input_sequences = []

# Create sequences of 6 words
# First 5 words = input, 6th word = target
for i in range(5, len(token_list)):
    n_gram_sequence = token_list[i-5:i+1]
    input_sequences.append(n_gram_sequence)

print(input_sequences[:5])


[[514, 4, 180, 4, 181, 1], [4, 180, 4, 181, 1, 515], [180, 4, 181, 1, 515, 56], [4, 181, 1, 515, 56, 64], [181, 1, 515, 56, 64, 202]]


In [12]:
token_list

[514,
 4,
 180,
 4,
 181,
 1,
 515,
 56,
 64,
 202,
 34,
 165,
 3,
 793,
 1,
 203,
 3,
 516,
 61,
 304,
 305,
 2,
 517,
 29,
 794,
 17,
 31,
 42,
 29,
 795,
 15,
 306,
 243,
 31,
 5,
 76,
 11,
 796,
 307,
 6,
 797,
 2,
 518,
 519,
 798,
 799,
 3,
 308,
 800,
 6,
 13,
 801,
 7,
 802,
 803,
 804,
 37,
 66,
 1,
 805,
 806,
 3,
 19,
 520,
 27,
 807,
 102,
 379,
 15,
 102,
 108,
 808,
 166,
 204,
 66,
 27,
 809,
 309,
 810,
 102,
 380,
 204,
 521,
 102,
 811,
 15,
 1,
 812,
 813,
 3,
 814,
 815,
 113,
 522,
 381,
 67,
 95,
 1,
 816,
 3,
 5,
 817,
 523,
 32,
 3,
 88,
 818,
 3,
 88,
 819,
 524,
 62,
 525,
 167,
 7,
 1,
 820,
 821,
 2,
 822,
 244,
 3,
 823,
 824,
 27,
 51,
 7,
 825,
 52,
 826,
 827,
 526,
 32,
 88,
 205,
 2,
 13,
 37,
 66,
 522,
 245,
 828,
 829,
 2,
 830,
 1,
 831,
 3,
 309,
 95,
 39,
 832,
 833,
 834,
 37,
 66,
 27,
 382,
 26,
 527,
 246,
 206,
 17,
 182,
 17,
 6,
 1,
 835,
 3,
 836,
 126,
 528,
 51,
 310,
 126,
 383,
 311,
 31,
 42,
 837,
 2,
 838,
 6,
 207,
 839,
 5,
 384,

PADDING

SPLIT X AND Y

In [13]:
max_sequence_length = 6

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1] # all sequences except last word/index
y = input_sequences[:, -1] # only last word/index

'''
[7, 121, 2, 1, 634, 158]
x= [7, 121, 2, 1, 634 ]
y = [158]
'''

# Convert target to one-hot encoding
y = to_categorical(y, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (9059, 5)
y shape: (9059, 2094)


In [14]:
y[0]

array([0., 1., 0., ..., 0., 0., 0.])

BUILD LSTM MODEL

In [15]:
model = Sequential()

# Correct input shape = sequence length - 1
model.add(Input(shape=(max_sequence_length - 1,))) # 5

# Embedding layer
model.add(Embedding(input_dim=total_words, output_dim=128))

# First LSTM layer
model.add(LSTM(150, return_sequences=True, dropout=0.2))

# Second LSTM layer
model.add(LSTM(100, dropout=0.2))

# Hidden Dense layer
model.add(Dense(100, activation='relu'))

# Output layer
model.add(Dense(total_words, activation='softmax')) #units=6032

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 5, 128)         │       268,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 5, 150)         │       167,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2094)           │       211,494 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 757,426 (2.89 MB)

 Trainable params: 757,426 (2.89 MB)

 Non-trainable params: 0 (0.00 B)

Model Training

In [16]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='loss',patience=3,restore_best_weights=True)

history = model.fit(X,y,epochs=20,batch_size=32,verbose=1,callbacks=[early_stop])


Epoch 1/20
284/284 ━━━━━━━━━━━━━━━━━━━━ 14s 33ms/step - accuracy: 0.0319 - loss: 6.7298
Epoch 2/20
284/284 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - accuracy: 0.0349 - loss: 6.3399
Epoch 3/20
284/284 ━━━━━━━━━━━━━━━━━━━━ 9s 26ms/step - accuracy: 0.0331 - loss: 6.2017
Epoch 4/20
284/284 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.0390 - loss: 6.0776
Epoch 5/20
284/284 ━━━━━━━━━━━━━━━━━━━━ 9s 31ms/step - accuracy: 0.0502 - loss: 5.9123
Epoch 6/20
284/284 ━━━━━━━━━━━━━━━━━━━━ 9s 27ms/step - accuracy: 0.0563 - loss: 5.7388
Epoch 7/20
284/284 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 0.0577 - loss: 5.5825
Epoch 8/20
284/284 ━━━━━━━━━━━━━━━━━━━━ 9s 31ms/step - accuracy: 0.0593 - loss: 5.4500
Epoch 9/20
284/284 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - accuracy: 0.0598 - loss: 5.3214
Epoch 10/20
284/284 ━━━━━━━━━━━━━━━━━━━━ 9s 31ms/step - accuracy: 0.0651 - loss: 5.1948
Epoch 11/20
284/284 ━━━━━━━━━━━━━━━━━━━━ 9s 32ms/step - accuracy: 0.0688 - loss: 5.0643
Epoch 12/20
284/284 ━━━━━━━━━━━━━━━━━━

Using the trained model for final predictions
TEMPERATURE + TOP-K SAMPLING

In [17]:
model.save("TextGenerationModel.keras")

In [18]:
def sample_with_temperature(preds, temperature=0.8, top_k=5):
    preds = np.asarray(preds).astype("float64")
    # [0.87,0.09,0.56,0.44,0.37,.............,0.89,0.32,...]

    # Select top k probabilities
    top_indices = np.argsort(preds)[-top_k:]
    # argsort : [0.09,0.32,0.37,0.44,0.56,0.87,0.89,........]
    # index of top k proabilities : [index]
    top_probs = preds[top_indices]
    #top k probs

    # Apply temperature scaling
    top_probs = np.log(top_probs + 1e-10) / temperature
    exp_probs = np.exp(top_probs)
    top_probs = exp_probs / np.sum(exp_probs)

    return np.random.choice(top_indices, p=top_probs)

Text Generation method

In [19]:
def generate_text(seed_text, next_words=20):
    output_text = seed_text
    generated_words = []

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_length - 1,
            padding='pre'
        ) # [0,0,0,189,45]

        predicted_probs = model.predict(token_list, verbose=0)[0] # [[0.89,0.07,.....,]] = [0.89,0.07,.....,]

        predicted_index = sample_with_temperature(
            predicted_probs,
            temperature=0.8,
            top_k=5
        )

        next_word = index_word.get(predicted_index, "")

        # avoid immediate repetition
        if next_word in generated_words[-3:]:
            continue

        generated_words.append(next_word)
        output_text += " " + next_word

    return output_text

Predictions

In [23]:
print(generate_text("So shaken he is ", next_words=15))

So shaken he is  i will king me let you make a brother out to hast sir how
